# 04 ? Frozen internal-test evaluation

**Objective:** Load each completed B0 best checkpoint and save paired per-image outputs.

**Experiment:** Historical Flat versus Shared-Hard baseline reconstruction, seed 42.

**Config:** `configs/experiments/flat/efficientnet_b0.yaml and configs/experiments/shared_hard/efficientnet_b0.yaml`

**Inputs:** Completed run summaries, frozen checkpoints, exact test manifest and raw test images.

**Outputs:** Prediction CSVs with logits/probabilities, endpoint/oracle metrics and hashed evaluation records.

**Mode:** Evaluation only; no optimization or checkpoint selection. Every input is loaded from disk; no other notebook's kernel state is required.

Use **Restart Kernel ? Run All**. Real training is disabled until the build handover is reviewed.


In [ ]:
from pathlib import Path
import sys, os
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "configs/protocol.yaml").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display, Markdown
from src.config import load_config
DATA_ROOT = os.environ.get("SKIN_CANCER_DATA_ROOT")
print("Dataset root:", Path(DATA_ROOT).expanduser().resolve() if DATA_ROOT else "NOT CONFIGURED ? set SKIN_CANCER_DATA_ROOT before image verification/training")


## Explicit frozen checkpoint inputs

The historical internal test is a consumed reproduction benchmark. Do not tune against its results. Oracle gating uses truth only as a diagnostic. Stage-2 logits are retained for every image; Stage-2 truth is missing for true NM.

In [ ]:
RUN_EVALUATION = False
DEVICE = "cuda"
RUN_IDS = ["flat_efficientnet_b0_seed42", "shared_hard_efficientnet_b0_seed42"]
from src.evaluation import evaluate
from src.statistics import load_evaluation
for run_id in RUN_IDS:
    if RUN_EVALUATION:
        display(evaluate(run_id, data_root=DATA_ROOT, device=DEVICE))
    elif (ROOT/"experiments/evaluations"/f"{run_id}_test"/"evaluation.json").exists():
        metadata, predictions = load_evaluation(f"{run_id}_test")
        display(metadata); display(predictions.head())
    else:
        print(run_id, "evaluation NOT RUN; complete and freeze training first")

## Summary and next step

Review the status and metrics displayed above. Missing artifacts mean **not run**, never a successful reproduction. Generated artifacts are listed in the output cells; scientific results remain separate from historical reference values.

**Next:** `05_flat_vs_hierarchical_comparison.ipynb`. Preserve completed run directories, prediction files and checkpoint backups before continuing.
